In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/dataset-training.csv")

In [ ]:
# inspection

df.head()
df.shape
df.columns
df.info()
df.describe()
df.isnull().sum()
df.duplicated().sum()


In [ ]:
# Remove the completely empty row:

df = df.dropna(how="all")

#remove CustomerID

df = df.drop(columns=["CustomerID"])

In [ ]:
# Make sure Churn has no missing values

df = df.dropna(subset=["Churn"])

print(df["Churn"].value_counts())

In [ ]:
 # Understand the target

df["Churn"].value_counts()
df["Churn"].value_counts(normalize=True)

df["Churn"].value_counts().plot(kind="bar")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
# Exploratory Data Analysis

# Churn vs Subscription

pd.crosstab(
    df["Subscription Type"],
    df["Churn"]
).plot(kind="bar")

plt.title("Churn by Subscription Type")
plt.xlabel("Subscription Type")
plt.ylabel("Customers")
plt.xticks(rotation=0)
plt.show()


# Churn vs Contract Length

pd.crosstab(
    df["Contract Length"],
    df["Churn"]
).plot(kind="bar")

plt.title("Churn by Contract Length")
plt.xlabel("Contract Length")
plt.ylabel("Customers")
plt.xticks(rotation=0)
plt.show()


# Churn vs Payment Delay

df.groupby("Churn")["Payment Delay"].mean()

df.boxplot(
    column="Payment Delay",
    by="Churn"
)

plt.title("Payment Delay vs Churn")
plt.suptitle("")
plt.xlabel("Churn")
plt.ylabel("Payment Delay")
plt.show()

In [ ]:
# Separate X and y (X = features, y = target)

# 80/20 split
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# 50-50 churn(0-1) use from 80 % of training dataset.

train_data = pd.concat(
    [X_train, y_train],
    axis=1
)

class_0 = train_data[train_data["Churn"] == 0]
class_1 = train_data[train_data["Churn"] == 1]

class_1_sample = class_1.sample(
    n=len(class_0),
    random_state=42
)

train_balanced = pd.concat(
    [class_0, class_1_sample]
)

train_balanced = train_balanced.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

X_train = train_balanced.drop(columns=["Churn"])
y_train = train_balanced["Churn"]

In [ ]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

print(X_train.isnull().sum())
print(X_test.isnull().sum())

print(y_train.isnull().sum())
print(y_test.isnull().sum())

In [ ]:
print(X_train.select_dtypes(include=["str"]).columns)

In [ ]:
numeric_features = [
    "Age",
    "Tenure",
    "Usage Frequency",
    "Support Calls",
    "Payment Delay",
    "Total Spend",
    "Last Interaction"
]

print("TRAIN")
print(X_train[numeric_features].describe().T)

print("\nTEST")
print(X_test[numeric_features].describe().T)



for col in ["Gender", "Subscription Type", "Contract Length"]:
    print("\n", col)

    print("TRAIN:")
    print(X_train[col].value_counts(normalize=True))

    print("TEST:")
    print(X_test[col].value_counts(normalize=True))

In [ ]:
# Preprocessing

In [ ]:
# handle categorical variables

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = X_train.select_dtypes(
    include=["str"]
).columns

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [ ]:
# Train model

# Logistic Regression

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000))
    ]
)

logistic_model.fit(X_train, y_train)

y_pred = logistic_model.predict(X_test)

In [ ]:
# Evaluate

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

In [ ]:
# confusion matrix 

from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred
)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Try Random Forest

from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                max_depth=15,
                n_jobs=-1
            )
        )
    ]
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [ ]:
# Evaluate
print(classification_report(y_test, rf_pred))

In [ ]:
# Decision Tree

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

decision_tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                random_state=42,
                max_depth=10
            )
        )
    ]
)

decision_tree_model.fit(X_train, y_train)

dt_pred = decision_tree_model.predict(X_test)


In [ ]:
print(classification_report(y_test, dt_pred))

In [ ]:
# test on the ORIGINAL test.csv

In [ ]:
test_df = pd.read_csv("../data/dataset-testing.csv")

X_kaggle_test = test_df.drop(
    columns=["CustomerID", "Churn"]
)

y_kaggle_test = test_df["Churn"]

In [ ]:
y_pred_lr = logistic_model.predict(X_kaggle_test)
y_pred_dt = decision_tree_model.predict(X_kaggle_test)
y_pred_rf = rf_model.predict(X_kaggle_test)

In [ ]:
from sklearn.metrics import classification_report

print("LOGISTIC REGRESSION")
print(classification_report(y_kaggle_test, y_pred_lr))

print("DECISION TREE")
print(classification_report(y_kaggle_test, y_pred_dt))

print("RANDOM FOREST")
print(classification_report(y_kaggle_test, y_pred_rf))